# Step 3 — DPO Alignment

## What this step does
Applies **Direct Preference Optimization (DPO)** on HH-RLHF preference pairs to further align
the federated model produced in Step 2. Trains a reference model alongside the policy model and
optimises the DPO loss to push chosen responses above rejected ones.

## Requirements from previous steps
- `./checkpoints/federated_model/` — produced by **Step 2**

## Produces
- `./checkpoints/dpo_model/` — DPO-aligned model weights


In [1]:
# Cell 2: GPU check + environment info
import subprocess
import sys
import os

print("=" * 60)
print("ENVIRONMENT INFO")
print("=" * 60)
print(f"Python: {sys.version}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        n_gpus = torch.cuda.device_count()
        print(f"Number of GPUs: {n_gpus}")
        for i in range(n_gpus):
            props = torch.cuda.get_device_properties(i)
            mem_gb = props.total_memory / 1024**3
            print(f"  GPU {i}: {props.name} ({mem_gb:.1f} GB)")
except ImportError:
    print("PyTorch not yet installed — will be installed in next cell.")

print("=" * 60)

ENVIRONMENT INFO
Python: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
PyTorch: 2.8.0+cu128
CUDA available: True
CUDA version: 12.8
Number of GPUs: 2
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (95.0 GB)
  GPU 1: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (95.0 GB)


In [2]:
%%bash
pip install -q transformers datasets peft flwr evaluate

In [3]:
# Imports + project config

import os, sys, warnings
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

warnings.filterwarnings("ignore")
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

MODEL_NAME  = "Qwen/Qwen2.5-7B"
LORA_CONFIG = LoraConfig(r=16, lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM)

MAX_SEQ_LEN = 512
BATCH_SIZE  = 2
DTYPE       = torch.bfloat16
BETA_DPO    = 0.05   # ← lowered from 0.1 for stronger correction after 10 rounds
DPO_EPOCHS  = 5      # ← 5 epochs for stronger alignment
N_ROUNDS    = 10     # ← match Step 2

BASE_DIR    = Path.home() / "federated-ethical-llm"
MODELS_DIR  = BASE_DIR / "models"
DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
for d in [MODELS_DIR, DATA_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAFETY_LORA_PATH = MODELS_DIR / "safety_lora"
FEDAVG_LORA_PATH = MODELS_DIR / f"fedavg_lora_{N_ROUNDS}r.pt"
DPO_LORA_PATH    = MODELS_DIR / f"dpo_lora_{N_ROUNDS}r"
STEP3_DONE_FLAG  = MODELS_DIR / f"step3_{N_ROUNDS}r_done.flag"
# Force GPU 1 — GPU 0 is occupied by other users
GPU_ID = 1 if torch.cuda.device_count() > 1 else 0
DEVICE = torch.device(f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu")

print(f"Config loaded. Using {N_ROUNDS}-round federated model.")
print(f"DPO epochs : {DPO_EPOCHS}  |  beta : {BETA_DPO}")
print(f"FEDAVG_LORA_PATH : {FEDAVG_LORA_PATH}")
print(f"DPO_LORA_PATH    : {DPO_LORA_PATH}")

2026-03-20 00:20:30.628161: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-20 00:20:30.638824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773984030.651622   11741 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773984030.655758   11741 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773984030.666059   11741 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Config loaded. Using 10-round federated model.
DPO epochs : 3  |  beta : 0.05
FEDAVG_LORA_PATH : /home/jovyan/federated-ethical-llm/models/fedavg_lora_10r.pt
DPO_LORA_PATH    : /home/jovyan/federated-ethical-llm/models/dpo_lora_10r


---
## Step 3 — DPO Alignment

After federated aggregation, we apply **Direct Preference Optimization (DPO)** to further align the model using the 500 preference pairs saved in Step 1.

DPO loss:
```
L_DPO = -E[ log σ( β * (log π(y_w|x) - log π_ref(y_w|x)) - β * (log π(y_l|x) - log π_ref(y_l|x)) ) ]
```

where `π_ref` is the safety-fine-tuned model (frozen) and `π` is the model being optimized.


In [4]:
# Shared helpers

def load_base_model_and_tokenizer(model_name=MODEL_NAME, for_training=True):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, padding_side="right")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    device_map = {"": GPU_ID} if for_training else {str(GPU_ID): "cuda:" + str(GPU_ID)}
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=DTYPE, device_map=device_map, trust_remote_code=True)
    model.config.use_cache = False
    model.enable_input_require_grads()
    return model, tokenizer

def apply_lora(model):
    model = get_peft_model(model, LORA_CONFIG)
    model.print_trainable_parameters()
    return model

print("Helpers defined.")

Helpers defined.


In [5]:
# DPO helper functions

def compute_log_probs(model, input_ids, attention_mask):
    """Compute per-sequence mean log-probability under model.
    Frees the large [B, T, V] logits/log_probs tensors immediately after
    gathering per-token log probs to minimise peak GPU memory.
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    shift_logits = outputs.logits[:, :-1, :].contiguous()
    del outputs  # free early

    shift_labels = input_ids[:, 1:].contiguous()
    shift_mask   = attention_mask[:, 1:].contiguous()

    log_probs = F.log_softmax(shift_logits, dim=-1)
    del shift_logits  # free the large [B, T, V] tensor immediately

    token_lp = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)
    del log_probs  # free immediately after gather

    token_lp = token_lp * shift_mask
    seq_lp   = token_lp.sum(dim=-1) / shift_mask.sum(dim=-1).clamp(min=1)
    return seq_lp


def dpo_loss(policy_chosen_lp, policy_rejected_lp,
             ref_chosen_lp, ref_rejected_lp, beta=BETA_DPO):
    """Standard DPO loss: -log σ(β * (π_chosen - π_ref_chosen - π_rejected + π_ref_rejected))"""
    pi_logratios  = policy_chosen_lp  - policy_rejected_lp
    ref_logratios = ref_chosen_lp     - ref_rejected_lp
    logits = beta * (pi_logratios - ref_logratios)
    return -F.logsigmoid(logits).mean()


print("compute_log_probs and dpo_loss defined.")

compute_log_probs and dpo_loss defined.


In [6]:
# run_dpo_alignment

def run_dpo_alignment(n_epochs=DPO_EPOCHS, lr=5e-5, dpo_batch_size=1):

    # Force all CUDA allocations (including PEFT adapter loading) onto GPU_ID
    torch.cuda.set_device(GPU_ID)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    print(f"DPO running on GPU {GPU_ID} ({torch.cuda.get_device_name(GPU_ID)})")

    print("Loading policy model ...")
    policy_base, dpo_tokenizer = load_base_model_and_tokenizer()
    policy_model = PeftModel.from_pretrained(policy_base, str(SAFETY_LORA_PATH), is_trainable=True)
    if FEDAVG_LORA_PATH.exists():
        agg = torch.load(str(FEDAVG_LORA_PATH), map_location="cpu")
        sd  = policy_model.state_dict()
        for k, v in agg.items():
            if k in sd:
                sd[k] = v.to(dtype=DTYPE)
        policy_model.load_state_dict(sd, strict=False)
        print(f"Aggregated weights loaded from {FEDAVG_LORA_PATH}")
    else:
        print(f"WARNING: {FEDAVG_LORA_PATH} not found — using safety LoRA only")

    # Gradient checkpointing: recompute activations during backward
    # instead of storing them — saves ~40% peak memory on policy model
    policy_model.gradient_checkpointing_enable()
    policy_model.enable_input_require_grads()
    print("Gradient checkpointing enabled on policy model.")

    print("Loading reference model (frozen) ...")
    ref_base, _ = load_base_model_and_tokenizer()
    ref_model   = PeftModel.from_pretrained(ref_base, str(SAFETY_LORA_PATH), is_trainable=False)
    for p in ref_model.parameters():
        p.requires_grad_(False)
    ref_model.eval()

    dpo_pairs      = torch.load(str(DATA_DIR / "dpo_pairs.pt"))
    chosen_texts   = [p[0] for p in dpo_pairs]
    rejected_texts = [p[1] for p in dpo_pairs]
    print(f"Loaded {len(dpo_pairs)} preference pairs.")

    chosen_enc   = dpo_tokenizer(chosen_texts,   truncation=True, max_length=MAX_SEQ_LEN,
                                 padding="max_length", return_tensors="pt")
    rejected_enc = dpo_tokenizer(rejected_texts, truncation=True, max_length=MAX_SEQ_LEN,
                                 padding="max_length", return_tensors="pt")

    optimizer = torch.optim.AdamW(
        [p for p in policy_model.parameters() if p.requires_grad], lr=lr)

    policy_model.train()
    print(f"\nStarting DPO — {n_epochs} epoch(s), {len(dpo_pairs)} pairs, beta={BETA_DPO}")
    print("Expected: loss starts ~0.69, should decrease below 0.60 with 3 epochs\n")

    for epoch in range(n_epochs):
        perm = torch.randperm(len(dpo_pairs))
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(range(0, len(dpo_pairs), dpo_batch_size), desc=f"Epoch {epoch+1}/{n_epochs}")

        for start in pbar:
            idx    = perm[start: start + dpo_batch_size]
            c_ids  = chosen_enc["input_ids"][idx].to(DEVICE)
            c_mask = chosen_enc["attention_mask"][idx].to(DEVICE)
            r_ids  = rejected_enc["input_ids"][idx].to(DEVICE)
            r_mask = rejected_enc["attention_mask"][idx].to(DEVICE)

            p_chosen_lp   = compute_log_probs(policy_model, c_ids, c_mask)
            p_rejected_lp = compute_log_probs(policy_model, r_ids, r_mask)
            with torch.no_grad():
                ref_chosen_lp   = compute_log_probs(ref_model, c_ids, c_mask)
                ref_rejected_lp = compute_log_probs(ref_model, r_ids, r_mask)

            loss = dpo_loss(p_chosen_lp, p_rejected_lp, ref_chosen_lp, ref_rejected_lp)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches  += 1
            pbar.set_postfix({"loss": f"{total_loss/n_batches:.4f}"})

        print(f"Epoch {epoch+1}: avg DPO loss = {total_loss/max(n_batches,1):.4f}")

    DPO_LORA_PATH.mkdir(parents=True, exist_ok=True)
    policy_model.save_pretrained(str(DPO_LORA_PATH))
    dpo_tokenizer.save_pretrained(str(DPO_LORA_PATH))
    print(f"\nDPO model saved to {DPO_LORA_PATH}")

    del policy_base, ref_base, policy_model, ref_model
    torch.cuda.empty_cache()


if STEP3_DONE_FLAG.exists():
    print(f"Step 3 ({N_ROUNDS}r) done — skipping. Delete {STEP3_DONE_FLAG.name} to re-run.")
else:
    run_dpo_alignment()
    STEP3_DONE_FLAG.touch()
    print("Step 3 complete.")


Loading policy model ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Aggregated weights loaded from /home/jovyan/federated-ethical-llm/models/fedavg_lora_10r.pt
Loading reference model (frozen) ...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loaded 500 preference pairs.

Starting DPO — 3 epoch(s), 500 pairs, beta=0.05
Expected: loss starts ~0.69, should decrease below 0.60 with 3 epochs



...

Epoch 1: avg DPO loss = 0.6961


Epoch 2/3:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 2: avg DPO loss = 0.6870


Epoch 3/3:   0%|          | 0/250 [00:00<?, ?it/s]

Epoch 3: avg DPO loss = 0.6531

DPO model saved to /home/jovyan/federated-ethical-llm/models/dpo_lora_10r
Step 3 complete.


---
## Step 4 — BOLD Evaluation: Demographic Parity

We evaluate four methods on the [BOLD dataset](https://huggingface.co/datasets/AlexaAI/bold):

| Method | Description |
|--------|-------------|
| **FedAvg baseline** | Aggregated LoRA weights without safety constraint |
| **Unconstrained LoRA** | LoRA fine-tuned on chosen data, no projection |
| **Biased full fine-tune** | All rejected data, no constraint |
| **SPA + safety constraint** | Proposed method (Steps 1–3 combined) |

For each method:
1. Generate 50 completions per demographic group in BOLD.
2. Score toxicity using the HuggingFace `evaluate` toxicity metric.
3. Compute **demographic parity gap** = `max_group_score − min_group_score`.

Results are plotted as a bar chart and printed as a table.
